# Rodada 5 — poucos blocos temporais

A enumeração elimina erro Monte Carlo, não a necessidade de pressupostos sobre blocos. IC exportados continuam sendo DK; não são IC bootstrap.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0,str(Path.cwd().parent/'testes2'))
from metodos import *
from bootstrap_blocos import wild_restrito
from IPython.display import display
e=Experimento(Path.cwd());d=e.d
for c in ['soja','milho']:
    for lag in [1,2,3]:d[f'expo_{c}_l{lag}']=d.groupby('id')['expo_'+c].shift(lag-1).where(d.ano.eq(d.groupby('id').ano.shift(lag-1)+lag-1))
lagcols=[f'expo_{c}_l{l}' for c in ['soja','milho'] for l in [1,2,3]]
referencias={};bs=[]

## 1. Núcleo estático amplo — dois destinos, duas restrições por destino

In [2]:
x=lagcols if False else ['expo_soja','expo_milho']
s=d.dropna(subset=['pp_agricultura','pp_pastagem',*x])
if False:s=s[s.ano.ge(2011)]
for uso in ['agricultura','pastagem']:
    ident='W01_'+uso
    r=e.linear(ident,'pp_'+uso,x,s,familia='referencias_inferencia',bioma_ano=False)
    for cultura in ['soja','milho']:
        pesos={f'expo_{cultura}_l{l}':1 for l in [1,2,3]} if False else {'expo_'+cultura:1}
        for tamanho in [3,2]:
            bs.append(wild_restrito(r,pesos,cultura,bloco=tamanho,bioma_ano=False,pasta=e.p))
    display(pd.DataFrame(bs).tail(4)[['id','contraste','bloco_anos','blocos','padroes','p_DK2','p_wild']])

,id,contraste,bloco_anos,blocos,padroes,p_DK2,p_wild
0,W01_agricultura,soja,3,8,256,0.088833,0.367188
1,W01_agricultura,soja,2,12,4096,0.088833,0.228027
2,W01_agricultura,milho,3,8,256,0.457940,0.679688
3,W01_agricultura,milho,2,12,4096,0.457940,0.337402


,id,contraste,bloco_anos,blocos,padroes,p_DK2,p_wild
4,W01_pastagem,soja,3,8,256,0.003208,0.085938
5,W01_pastagem,soja,2,12,4096,0.003208,0.046387
6,W01_pastagem,milho,3,8,256,0.039979,0.140625
7,W01_pastagem,milho,2,12,4096,0.039979,0.096680


## 2. Somas de três lags amplos — dois destinos, duas restrições por destino

In [3]:
x=lagcols if True else ['expo_soja','expo_milho']
s=d.dropna(subset=['pp_agricultura','pp_pastagem',*x])
if False:s=s[s.ano.ge(2011)]
for uso in ['agricultura','pastagem']:
    ident='W02_'+uso
    r=e.linear(ident,'pp_'+uso,x,s,familia='referencias_inferencia',bioma_ano=False)
    for cultura in ['soja','milho']:
        pesos={f'expo_{cultura}_l{l}':1 for l in [1,2,3]} if True else {'expo_'+cultura:1}
        for tamanho in [3,2]:
            bs.append(wild_restrito(r,pesos,cultura,bloco=tamanho,bioma_ano=False,pasta=e.p))
    display(pd.DataFrame(bs).tail(4)[['id','contraste','bloco_anos','blocos','padroes','p_DK2','p_wild']])

,id,contraste,bloco_anos,blocos,padroes,p_DK2,p_wild
8,W02_agricultura,soja,3,8,256,0.073480,0.250000
9,W02_agricultura,soja,2,11,2048,0.073480,0.125977
10,W02_agricultura,milho,3,8,256,0.651802,0.929688
11,W02_agricultura,milho,2,11,2048,0.651802,0.730469


,id,contraste,bloco_anos,blocos,padroes,p_DK2,p_wild
12,W02_pastagem,soja,3,8,256,2.803419e-07,0.007812
13,W02_pastagem,soja,2,11,2048,2.803419e-07,0.001953
14,W02_pastagem,milho,3,8,256,4.368875e-02,0.078125
15,W02_pastagem,milho,2,11,2048,4.368875e-02,0.125977


## 3. Somas com FE bioma×ano — dois destinos, duas restrições por destino

In [4]:
x=lagcols if True else ['expo_soja','expo_milho']
s=d.dropna(subset=['pp_agricultura','pp_pastagem',*x])
if False:s=s[s.ano.ge(2011)]
for uso in ['agricultura','pastagem']:
    ident='W03_'+uso
    r=e.linear(ident,'pp_'+uso,x,s,familia='referencias_inferencia',bioma_ano=True)
    for cultura in ['soja','milho']:
        pesos={f'expo_{cultura}_l{l}':1 for l in [1,2,3]} if True else {'expo_'+cultura:1}
        for tamanho in [3,2]:
            bs.append(wild_restrito(r,pesos,cultura,bloco=tamanho,bioma_ano=True,pasta=e.p))
    display(pd.DataFrame(bs).tail(4)[['id','contraste','bloco_anos','blocos','padroes','p_DK2','p_wild']])

,id,contraste,bloco_anos,blocos,padroes,p_DK2,p_wild
16,W03_agricultura,soja,3,8,256,0.053637,0.257812
17,W03_agricultura,soja,2,11,2048,0.053637,0.151367
18,W03_agricultura,milho,3,8,256,0.396370,0.828125
19,W03_agricultura,milho,2,11,2048,0.396370,0.665039


,id,contraste,bloco_anos,blocos,padroes,p_DK2,p_wild
20,W03_pastagem,soja,3,8,256,0.696724,0.664062
21,W03_pastagem,soja,2,11,2048,0.696724,0.699219
22,W03_pastagem,milho,3,8,256,0.001906,0.015625
23,W03_pastagem,milho,2,11,2048,0.001906,0.031250


## 4. Somas na janela 2011–2024 — dois destinos, duas restrições por destino

In [5]:
x=lagcols if True else ['expo_soja','expo_milho']
s=d.dropna(subset=['pp_agricultura','pp_pastagem',*x])
if True:s=s[s.ano.ge(2011)]
for uso in ['agricultura','pastagem']:
    ident='W04_'+uso
    r=e.linear(ident,'pp_'+uso,x,s,familia='referencias_inferencia',bioma_ano=False)
    for cultura in ['soja','milho']:
        pesos={f'expo_{cultura}_l{l}':1 for l in [1,2,3]} if True else {'expo_'+cultura:1}
        for tamanho in [3,2]:
            bs.append(wild_restrito(r,pesos,cultura,bloco=tamanho,bioma_ano=False,pasta=e.p))
    display(pd.DataFrame(bs).tail(4)[['id','contraste','bloco_anos','blocos','padroes','p_DK2','p_wild']])

,id,contraste,bloco_anos,blocos,padroes,p_DK2,p_wild
24,W04_agricultura,soja,3,5,32,0.006878,0.125000
25,W04_agricultura,soja,2,7,128,0.006878,0.046875
26,W04_agricultura,milho,3,5,32,0.000101,0.062500
27,W04_agricultura,milho,2,7,128,0.000101,0.015625


,id,contraste,bloco_anos,blocos,padroes,p_DK2,p_wild
28,W04_pastagem,soja,3,5,32,0.006628,0.125000
29,W04_pastagem,soja,2,7,128,0.006628,0.234375
30,W04_pastagem,milho,3,5,32,0.754562,0.937500
31,W04_pastagem,milho,2,7,128,0.754562,0.781250


## 5. Multiplicidade e verificações

In [6]:
b=pd.DataFrame(bs);assert len(b)==32
b['p_wild_holm']=np.nan
for tamanho,ix in b.groupby('bloco_anos').groups.items():
    assert len(ix)==16
    b.loc[ix,'p_wild_holm']=multipletests(b.loc[ix,'p_wild'].fillna(1),method='holm')[1]
    b.loc[b.index.isin(ix)&b.p_wild.isna(),'p_wild_holm']=np.nan
assert b.reestimacao_direta_verificada.all()
b.to_csv(e.p/'BOOTSTRAP.csv',index=False)
e.salvar('referencias')
display(b[['id','contraste','bloco_anos','blocos','p_wild','p_wild_holm']])
print('Não há independência/simetria dos blocos demonstrada. P não significativo não é equivalência.')

,id,contraste,bloco_anos,blocos,p_wild,p_wild_holm
0,W01_agricultura,soja,3,8,0.367188,1.000000
1,W01_agricultura,soja,2,12,0.228027,1.000000
2,W01_agricultura,milho,3,8,0.679688,1.000000
3,W01_agricultura,milho,2,12,0.337402,1.000000
4,W01_pastagem,soja,3,8,0.085938,1.000000
5,W01_pastagem,soja,2,12,0.046387,0.603027
6,W01_pastagem,milho,3,8,0.140625,1.000000
7,W01_pastagem,milho,2,12,0.096680,1.000000
8,W02_agricultura,soja,3,8,0.250000,1.000000
9,W02_agricultura,soja,2,11,0.125977,1.000000


Não há independência/simetria dos blocos demonstrada. P não significativo não é equivalência.
